# **Reflection Agent with External Knowledge Integration**

This agent is designed to not just answer a question, but to critique its own answer, identify weaknesses, use tools to find more information, and then revise its answer to be more accurate and comprehensive.


In [1]:
%%capture
%pip install openai
%pip install  --upgrade langgraph

In [ ]:
%pip install -U langchain-ibm langchain langchain-core langchain-community langgraph langchain-openai


In [ ]:
import os
import json
import getpass
from typing import List, Dict
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage, BaseMessage
from langchain_community.utilities.tavily_search import TavilySearchAPIWrapper
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_openai import ChatOpenAI
from langgraph.graph import END, StateGraph  # MessageGraph - Deprecated

In [3]:
from langchain_openai import ChatOpenAI
from langchain_ibm import ChatWatsonx


openai_api_key = os.getenv("OPENAI_KEY")
whatsonx_api_key = os.getenv("WATSONX_APIKEY")
ibm_project_id = os.getenv("PROJECT_ID")

openai_llm = ChatOpenAI(
    model="gpt-4.1-nano",
    api_key = openai_api_key,
)
watsonx_llm = ChatWatsonx(
    model_id="ibm/granite-4-h-small",
    url="https://us-south.ml.cloud.ibm.com",
    project_id=ibm_project_id,
    api_key=whatsonx_api_key,
)

In [4]:
from langchain_community.tools import TavilySearchResults
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
tavily_tool=TavilySearchResults(max_results=1)
sample_query = "healthy breakfast recipes"
#search_results = tavily_tool.invoke(sample_query)
#print(search_results)

## LLM and Prompting

In [6]:
question="Any ideas for a healthy breakfast"
response=openai_llm.invoke(question).content
print(response)

Certainly! Here are some healthy breakfast ideas to start your day:

1. **Oatmeal with Fresh Fruit**  
   - Whole rolled oats topped with berries, banana slices, and a sprinkle of nuts or seeds.

2. **Greek Yogurt Parfait**  
   - Greek yogurt layered with granola, fresh fruit, and a drizzle of honey.

3. **Veggie Omelette**  
   - Eggs whisked with spinach, tomatoes, peppers, and onions. Serve with a slice of whole-grain toast.

4. **Smoothie Bowl**  
   - Blend frozen berries, banana, and spinach with a splash of almond milk. Top with granola, chia seeds, and sliced fruit.

5. **Whole Grain Toast with Avocado and Egg**  
   - Mashed avocado spread on toast topped with a poached or boiled egg.

6. **Chia Seed Pudding**  
   - Soak chia seeds in almond milk overnight, then top with fresh fruit and nuts in the morning.

7. **Buckwheat or Quinoa Porridge**  
   - Cooked whole grains topped with nuts, seeds, and a little honey or maple syrup.

8. **Fresh Fruit and Nut Butter**  
   - Slic

In [7]:
question="Any ideas for a healthy breakfast"
response=watsonx_llm.invoke(question).content
print(response)

Certainly, here are a few suggestions for a healthy breakfast:

1. **Oatmeal with Fresh Fruit and Nuts**: Oats are high in fiber and can keep you feeling full throughout the morning. Adding fruits like blueberries or strawberries will give you a boost of vitamins and antioxidants. A sprinkle of nuts or seeds adds healthy fats and protein.

2. **Greek Yogurt with Granola and Honey**: Greek yogurt is a great source of protein and calcium. Adding a dollop of honey and some granola will give it a sweet kick and crunch.

3. **Avocado Toast with Poached Egg**: Avocado is high in heart-healthy fats. A poached egg on top will provide a good amount of protein to keep you full.

4. **Smoothie Bowl**: Blend together your favorite fruits with some Greek yogurt and a splash of almond milk. Pour into a bowl and top with nuts, seeds, or granola for added fiber and protein.

5. **Quinoa Porridge**: Quinoa is a complete protein and can be cooked with milk or a non-dairy alternative for a protein-packed

## Crafting the Agent's Persona and Logic

In [8]:
prompt_template = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are Dr. Paul Saladino, "Carnivore MD," advocating for animal-based nutrition and challenging plant-centric dietary dogma. Focus on the superior bioavailability of animal foods and the potential toxicity of plant compounds such as oxalates, lectins, and phytates.

        Your response must follow these steps:
        1. {first_instruction}
        2. Present the evolutionary and biochemical rationale for animal-based nutrition, emphasizing nutrient density, bioavailability, and the absence of plant antinutrients.
        3. Challenge conventional "plants are healthy" narratives with mechanistic evidence about plant defense compounds and their impact on human physiology.
        4. Reflect and critique your answer. Be rigorous about evolutionary consistency and nutrient science.
        5. After the reflection, **list 1-3 search queries separately** for researching antinutrients, bioavailability studies, or evolutionary nutrition. Do not include them inside the reflection.

        Focus on: organ meats as superfoods, the carnivore elimination protocol, and why "fiber is unnecessary" based on human evolutionary biology.
        """
    ),
    MessagesPlaceholder(variable_name="messages"),
    (
        "system", 
        "Answer the user's question above using the required format, emphasizing the superiority of animal-based nutrition."
    ),
])

## Defining the Responder based on the user's question and the persona we defined in the prompt.

In [9]:
first_responder_prompt = prompt_template.partial(first_instruction="Provide a detailed ~250 word answer")
temp_chain = first_responder_prompt| openai_llm
response = temp_chain.invoke({"messages": [HumanMessage(content=question)]})
print(response.content)

A truly optimal breakfast, grounded in evolutionary and biochemical principles, would center around nutrient-dense, bioavailable animal foods. Organ meats, such as liver, kidneys, and heart, are unparalleled in their nutrient density, providing essential vitamins like B12, A, and folate, along with vital minerals such as zinc and copper—nutrients more efficiently absorbed from animal tissues than from plant sources. Consuming these as part of a carnivore-based breakfast ensures a comprehensive intake of amino acids, heme iron, and other cofactors crucial for cellular function and metabolic health.

From an evolutionary standpoint, humans adapted to predominantly animal-based diets for millions of years, with bioavailability being a key factor. Animal proteins and fats offer highly digestible, readily accessible nutrients without the presence of plant antinutrients—namely oxalates, lectins, and phytates—that can impair mineral absorption or trigger inflammation. These compounds serve pl

## Structuring the Agent's Output: Data Models

In [ ]:
class Reflection(BaseModel):
	missing: str = Field(description="What information is missing")
	superfluous: str = Field(description="What information is unnecessary")

class AnswerQuestion(BaseModel):
	answer: str = Field(description="Main response to the question")
	reflection: Reflection = Field(description="Self-critique of the answer")
	search_queries: List[str] = Field(description="Queries for additional research")

## Binding Tools to the Responder

In [ ]:
initial_chain = first_responder_prompt| openai_llm.bind_tools(tools=[AnswerQuestion])
response=initial_chain.invoke({"messages":[HumanMessage(question)]})
print("---Full Structured Output---")
print(response.tool_calls)

In [ ]:
answer_content = response.tool_calls[0]['args']['answer']
print("---Initial Answer---")
print(answer_content)

In [ ]:
Reflection_content = response.tool_calls[0]['args']['reflection']
print("---Reflection Answer---")
print(Reflection_content)

In [ ]:
search_queries = response.tool_calls[0]['args']['search_queries']
print("---Search Queries---")
print(search_queries)

## Tool Execution

In [ ]:
response_list=[]
response_list.append(HumanMessage(content=question))
response_list.append(response)

In [ ]:
tool_call=response.tool_calls[0]
search_queries = tool_call["args"].get("search_queries", [])
print(search_queries)

In [ ]:
tavily_tool=TavilySearchResults(max_results=3)



def execute_tools(state: List[BaseMessage]) -> List[BaseMessage]:
    last_ai_message = state[-1]
    tool_messages = []
    for tool_call in last_ai_message.tool_calls:
        if tool_call["name"] in ["AnswerQuestion", "ReviseAnswer"]:
            call_id = tool_call["id"]
            search_queries = tool_call["args"].get("search_queries", [])
            query_results = {}
            for query in search_queries:
                result = tavily_tool.invoke(query)
                query_results[query] = result
            tool_messages.append(ToolMessage(
                content=json.dumps(query_results),
                tool_call_id=call_id)
            )
    return tool_messages

In [ ]:
tool_response = execute_tools(response_list)
# Use .extend() to add all tool messages from the list
response_list.extend(tool_response)

In [ ]:
tool_response

In [ ]:
response_list

## Defining the Revisor

The **Revisor** is the final piece of the Reflection loop. Its job is to take the original answer, the self-critique, and the new information from the tool search, and then generate an improved, more evidence-based response.

It was created a new set of instructions (`revise_instructions`) that guide the Revisor. These instructions emphasize:
- Incorporating the critique.
- Adding numerical citations from the research.
- Distinguishing between correlation and causation.
- Adding a "References" section.

In [ ]:
revise_instructions = """Revise your previous answer using the new information, applying the rigor and evidence-based approach of Dr. David Attia.
- Incorporate the previous critique to add clinically relevant information, focusing on mechanistic understanding and individual variability.
- You MUST include numerical citations referencing peer-reviewed research, randomized controlled trials, or meta-analyses to ensure medical accuracy.
- Distinguish between correlation and causation, and acknowledge limitations in current research.
- Address potential biomarker considerations (lipid panels, inflammatory markers, and so on) when relevant.
- Add a "References" section to the bottom of your answer (which does not count towards the word limit) in the form of:
- [1] https://example.com
- [2] https://example.com
- Use the previous critique to remove speculation and ensure claims are supported by high-quality evidence. Keep response under 250 words with precision over volume.
- When discussing nutritional interventions, consider metabolic flexibility, insulin sensitivity, and individual response variability.
"""
revisor_prompt = prompt_template.partial(first_instruction=revise_instructions)

## Structuring the Revisor's Output

Binding the new tool to the revisor chain:

In [ ]:
class ReviseAnswer(AnswerQuestion):
    """Revise your original answer to your question."""
    references: List[str] = Field(description="Citations motivating your updated answer.")
revisor_chain = revisor_prompt | openai_llm.bind_tools(tools=[ReviseAnswer])